# WANDS: product-search relevance comparison

Use a **fresh GPU runtime** and run all cells. This compares **DenseOn, LateOn (ColBERT), ModernBERT cross-encoder, and BM25** using human WANDS relevance judgments.

The default uses all **480 queries and 42,994 products**. DenseOn/LateOn are the paired 149M models; the CE is separately trained. No WANDS training or label-derived candidates. This is a quality experiment, not a serving-latency benchmark.


In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

def run_logged(command, *, cwd=None, log_path):
    """Forward child stdout/stderr through notebook output and keep the failure tail."""
    import collections
    import subprocess
    import sys
    from pathlib import Path

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = collections.deque(maxlen=80)
    print(f'Python: {sys.version.split()[0]} | executable: {sys.executable}', flush=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n--- New invocation ---\n')
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                              errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if returncode:
        raise RuntimeError(
            f'Command exited with status {returncode}. Full log: {log_path}\n'
            + ''.join(tail))
    return returncode

drive.mount('/content/drive')
REPO = Path('/content/ras-wands')
BRANCH = 'codex/colbert-muvera-baselines'
LOGS = Path('/content/drive/MyDrive/ras_wands_logs')
if not REPO.exists():
    run_logged(['git', 'clone', '--branch', BRANCH, '--single-branch',
                'https://github.com/hanialshater/ras.git', str(REPO)],
               log_path=LOGS / 'setup.log')
else:
    run_logged(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH],
               log_path=LOGS / 'setup.log')
print(subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True))
run_logged([sys.executable, '-m', 'pip', 'install', '-e',
            str(REPO) + '[dev,wands]'],
           log_path=LOGS / 'install.log')


In [ ]:
import os
os.chdir(REPO)
os.environ['PYTHONPATH'] = str(REPO / 'src') + ':' + str(REPO)
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['PYLATE_SCORES_BACKEND'] = 'torch'
# Imports must succeed so the native integration test cannot silently skip.
run_logged([sys.executable, '-c', 'import torch, pylate, sentence_transformers, ir_measures; assert torch.cuda.is_available(), "Select a GPU runtime"'], cwd=REPO, log_path=LOGS / 'environment.log')
run_logged([sys.executable, '-m', 'pytest', '-q', 'tests/test_wands_comparison.py', 'tests/test_upstream_nanobeir.py'], cwd=REPO, log_path=LOGS / 'tests.log')


## Configuration and source audit

The original WANDS files and model revisions are pinned. Product text includes name, category, attributes, then description. Model input caps are 512; the cross-encoder cap includes the query. Exact=2, Partial=1, Irrelevant=0 with **linear nDCG gains**. Duplicate labels collapse; conflicting pairs take the minimum grade and are listed in the audit. All queries, including those without positives, remain in the evaluation.

Set `QUERY_LIMIT = 50` for an exploratory run; the full corpus is still searched. Use a new `RUN` directory whenever changing configuration/code. Completed document score shards and CE queries resume automatically.


In [ ]:
RUN = Path('/content/drive/MyDrive/ras_wands_denseon_lateon_seed7_v1')
QUERY_LIMIT = 0  # All 480. Use 50 for an exploratory subset.
POOL_SIZE = 100  # Union of top 100 dense and top 100 BM25: at most 200 products.
BASE = [sys.executable, '-u', '-m', 'experiments.wands_comparison',
        '--output-dir', str(RUN), '--queries', str(QUERY_LIMIT),
        '--pool-size', str(POOL_SIZE), '--seed', '7']
run_logged(BASE + ['--phase', 'data'], cwd=REPO, log_path=LOGS / 'data.log')
import json
import pandas as pd
from IPython.display import display
audit = json.loads((RUN / 'data_audit.json').read_text())
print(json.dumps({k: v for k, v in audit.items() if k != 'conflicting_pairs'}, indent=2))
print('Conflicting label pairs:', len(audit['conflicting_pairs']))


## Verify real model interfaces

Before the long run, encode two real queries and four real products through all three neural models. This small check produces no relevance claims. A failure stops the notebook with a saved log.


In [ ]:
run_logged(BASE + ['--phase', 'smoke'], cwd=REPO, log_path=LOGS / 'smoke.log')


## Run the comparison

Dense, ColBERT and BM25 search the full corpus. Then all four methods rank the **same union of dense and BM25 candidates**. No positive products are injected. ColBERT uses PyLate's native MaxSim function, grouping equal-length document embeddings to prevent padding from changing scores.

Progress prints after every 256 products and every CE query. Source files and completed scores persist on Drive. These offline durations do **not** measure serving latency, index build time or per-method memory.


In [ ]:
run_logged(BASE + ['--phase', 'compare'], cwd=REPO, log_path=LOGS / 'comparison.log')


In [ ]:
quality = pd.read_csv(RUN / 'quality.csv')
print('Full-corpus retrieval and identical-pool ranking; full-corpus qrels in both scopes')
display(quality)
print('Paired query bootstrap differences: method A minus B; scopes kept separate')
display(pd.read_csv(RUN / 'paired_ndcg.csv'))
print('Candidate coverage of judged Exact products')
display(pd.read_csv(RUN / 'pool_coverage.csv').describe())
print('Actual loaded models and input settings')
for method in ['dense', 'colbert', 'ce']:
    print(method, (RUN / method / 'model.json').read_text())
print((RUN / 'scope.json').read_text())


## Export results

`ndcg10` uses graded human judgments. Exact-only MRR@10, precision@10 and recall@100 are reported separately. Unjudged products count as zero in standard metrics; `judged_at10` exposes how much of the top ten has human judgments. Recall is over **known judged Exact products**, not exhaustive relevance. The shared pool always uses full-corpus metric denominators.

The ZIP contains the dataset, audit, model settings, rankings, score shards and results. Inspect query-level differences before drawing conclusions. WANDS is home furnishings, not a fashion-specific or multilingual benchmark.


In [ ]:
import zipfile
from google.colab import files
archive = RUN.parent / (RUN.name + '.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in RUN.rglob('*'):
        if path.is_file():
            z.write(path, str(path.relative_to(RUN)))
    for path in LOGS.glob('*.log'):
        z.write(path, 'logs/' + path.name)
print(archive)
files.download(str(archive))
